# 01 — Quality Gate 1

Що демонструє ноутбук (п. 2.1 ТЗ):
1. **Евристичний бейзлайн без ML** — Accuracy / Recall / Precision на всьому наборі.
2. **Спрацювання перевірки конфігурації** на навмисно зіпсованому файлі (захищена ознака у `features`).
3. **Дві конфігурації** відрізняються одним рядком.
4. **Навчання препроцесора** лише на тренувальній підвибірці, **аудиторський слід** імпутера і таблиця метаданих (п. 5.3).
5. `transform` **не переобчислює** частку пропусків.
6. **Протокол верифікації** `run_verification.py`.

> Класифікатор у цій роботі **не навчається**. `fit` викликається тільки на препроцесорі.


In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)

# Стиль графіків: один акцентний колір, приглушені осі й сітка
ACCENT, INK, MUTED, SURFACE = "#2a78d6", "#0b0b0b", "#52514e", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": "#d0cfca", "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.titlecolor": INK, "axes.titlesize": 10, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.grid": True, "grid.color": "#ecebe7", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110,
})

In [2]:
from src.data import load_data_config, load_dataset, split_xy
dc = load_data_config()
df = load_dataset(dc)                     # сирі дані + MCAR-пропуски з config/data.yaml (seed=42)
X, y = split_xy(df, dc)
print(df.shape, "| частка відтоку:", round(y.mean(), 4))
print("Параметри інжектування:", dc["missing_injection"])

(5630, 20) | частка відтоку: 0.1684
Параметри інжектування: {'mechanism': 'MCAR', 'seed': 42, 'columns': {'CashbackAmount': 0.2, 'PreferedOrderCat': 0.15}}


## 1. Евристичний бейзлайн (Non-ML Benchmark)

Правило менеджера з утримання клієнтів: **«клієнт новий (Tenure ≤ 1 міс.) АБО скаржився минулого місяця → ризик відтоку»**.
Дві найочевидніші бізнес-причини піти: ще не звик до сервісу або вже незадоволений.
Позитивний клас — `Churn = 1`. Пропуск у `Tenure` → умова «новий» вважається невиконаною (порівняння з NaN = False).


In [3]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix

y_true = y
y_heuristic = ((df["Tenure"] <= 1) | (df["Complain"] == 1)).astype(int)
y_majority = pd.Series(0, index=df.index)          # «ніхто не піде» — для контексту Accuracy

def scores(pred):
    return {"Accuracy": accuracy_score(y_true, pred), "Recall": recall_score(y_true, pred, zero_division=0),
            "Precision": precision_score(y_true, pred, zero_division=0), "F1": f1_score(y_true, pred, zero_division=0),
            "flagged_share": pred.mean()}

benchmark = pd.DataFrame({"heuristic: Tenure<=1 OR Complain": scores(y_heuristic),
                          "reference: always 0": scores(y_majority)}).T.round(3)
benchmark

,Accuracy,Recall,Precision,F1,flagged_share
heuristic: Tenure<=1 OR Complain,0.699,0.848,0.341,0.487,0.418
reference: always 0,0.832,0.000,0.000,0.000,0.000


In [4]:
pd.DataFrame(confusion_matrix(y_true, y_heuristic), index=["true 0", "true 1"], columns=["pred 0", "pred 1"])

,pred 0,pred 1
true 0,3130,1552
true 1,144,804


In [5]:
# Частка клієнтів, яких правило позначає як «ризик», за захищеною ознакою (довідково, не метрика справедливості моделі)
pd.DataFrame({"flagged_share": y_heuristic.groupby(df["CityTier"]).mean(),
              "churn_rate": y_true.groupby(df["CityTier"]).mean()}).round(3)

,flagged_share,churn_rate
CityTier,,
1,0.413,0.145
2,0.380,0.198
3,0.436,0.214


**Висновок.** Евристика ловить ~85% клієнтів, що йдуть (Recall), але лише ~34% позначених справді йдуть (Precision).
Accuracy 0.70 — нижча за тривіальне «ніхто не піде» (0.83): при дисбалансі 17/83 Accuracy не є інформативною метрикою.
Тому ML Metric формулюється через Recall/Precision класу `Churn = 1` (див. README, ML Canvas).


## 2. Перевірка конфігурації: зіпсований файл

Беремо робочий `config_lr.yaml`, додаємо захищену ознаку `CityTier` у `features` і пробуємо створити конфігуратор.
Очікування: `ValueError` у конструкторі — **до** збирання `ColumnTransformer` і до будь-яких обчислень.


In [6]:
import tempfile, yaml
from src.dynamic_pipeline_builder import DynamicPipelineBuilder

with open(ROOT / "config" / "config_lr.yaml", encoding="utf-8") as f:
    broken = yaml.safe_load(f)
broken["features"]["categorical_ordinal"].append("CityTier")      # навмисна помилка

with tempfile.NamedTemporaryFile("w", suffix=".yaml", delete=False, encoding="utf-8") as tmp:
    yaml.safe_dump(broken, tmp, allow_unicode=True)

try:
    DynamicPipelineBuilder(tmp.name)
    print("❌ Конфігурацію прийнято — перевірка не спрацювала")
except ValueError as e:
    print("✅ Збирання заблоковано:", e)

✅ Збирання заблоковано: Захищені ознаки потрапили у features: {'CityTier'}


In [7]:
# Додаткові вхідні ворота (validate_schema): непридатна схема ключів model_family
with open(ROOT / "config" / "config_lr.yaml", encoding="utf-8") as f:
    broken2 = yaml.safe_load(f)
broken2["model_family"] = "simple"
with tempfile.NamedTemporaryFile("w", suffix=".yaml", delete=False, encoding="utf-8") as tmp2:
    yaml.safe_dump(broken2, tmp2, allow_unicode=True)
try:
    DynamicPipelineBuilder(tmp2.name)
except ValueError as e:
    print("✅ Збирання заблоковано:", e)

✅ Збирання заблоковано: model_family='simple' не описано в transform_policy (є: ['additive', 'partitioning'])


## 3. Дві конфігурації — один рядок різниці

In [8]:
import difflib
lr = (ROOT / "config" / "config_lr.yaml").read_text(encoding="utf-8").splitlines()
rf = (ROOT / "config" / "config_rf.yaml").read_text(encoding="utf-8").splitlines()
print("\n".join(difflib.unified_diff(lr, rf, "config_lr.yaml", "config_rf.yaml", lineterm="", n=0)))

--- config_lr.yaml
+++ config_rf.yaml
@@ -3 +3 @@
-model_family: additive          # additive | partitioning  (п. 3.4.4: спосіб комбінування ознак)
+model_family: partitioning        # additive | partitioning  (п. 3.4.4: спосіб комбінування ознак)


In [9]:
builders = {name: DynamicPipelineBuilder(ROOT / "config" / f"config_{name}.yaml") for name in ("lr", "rf")}
builders["lr"].policy_summary()

,group,additive,partitioning
0,numeric_continuous,log1p → standard_scaler,winsorize
1,numeric_binary,standard_scaler,passthrough
2,categorical_nominal,woe,one_hot
3,categorical_ordinal,ordinal_encoder → standard_scaler,ordinal_encoder


In [10]:
builders["lr"].build()       # ненавчений ColumnTransformer: до даних не звертається

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric_continuous', ...), ('numeric_binary', ...), ...]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights fo

## 4. Навчання препроцесора на тренувальній підвибірці

Стратифіковане розбиття 80/20 з `random_state=42` — потрібне лише для того, щоб показати навчання препроцесора на train
(п. 5.3: частка пропусків береться з того, що імпутер *побачив на тренувальній підвибірці*).
Препроцесор навчається **лише** на `X_train`; `y_train` потрібен для WoE-кодування.


In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

fitted = {}
for name, b in builders.items():
    ct = b.build().set_output(transform="pandas")
    Z_train = ct.fit_transform(X_train, y_train)
    Z_test = ct.transform(X_test)
    fitted[name] = ct
    protected_out = [c for c in Z_train.columns if any(p in c for p in b.config["protected_attributes"])]
    print(f"{name}: train {Z_train.shape}, test {Z_test.shape}, NaN після обробки: {int(Z_test.isna().sum().sum())}, "
          f"захищені ознаки у виході: {protected_out or 'немає'}")

lr: train (4504, 17), test (1126, 17), NaN після обробки: 0, захищені ознаки у виході: немає
rf: train (4504, 29), test (1126, 29), NaN після обробки: 0, захищені ознаки у виході: немає


In [12]:
fitted["lr"].transform(X_test).head()

,Tenure,WarehouseToHome,HourSpendOnApp,NumberOfDeviceRegistered,NumberOfAddress,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,CashbackAmount__missing,Complain,PreferredLoginDevice,PreferredPaymentMode,PreferedOrderCat,MaritalStatus,SatisfactionScore
4006,0.866261,-0.850640,1.375983,0.361344,-0.322372,1.456032,2.014642,1.899627,1.026611,0.266435,-0.502497,-0.623912,0.223257,-0.111055,-0.510322,-0.411807,-1.493573
1969,1.267462,-0.130511,0.177516,0.361344,1.414275,0.740269,-1.514994,-0.967923,-0.121419,0.796578,-0.502497,1.602791,-0.101676,0.416799,-0.014827,-0.411807,-1.493573
4235,-0.220799,0.155278,1.375983,0.361344,-0.322372,2.080153,0.477750,-0.194896,1.175769,-0.000925,-0.502497,-0.623912,-0.101676,-0.111055,-0.510322,0.583508,-1.493573
105,1.230119,-1.076136,-1.367577,0.361344,1.414275,-0.098756,-1.514994,-0.194896,-0.528690,-0.928290,-0.502497,-0.623912,-0.101676,-0.111055,0.617617,0.583508,-0.771089
2295,-0.401204,-1.076136,-1.367577,-0.522429,-0.938457,0.990699,2.470494,2.100499,0.670826,0.597460,-0.502497,1.602791,0.223257,-0.111055,-0.014827,-0.411807,-0.048604


In [13]:
fitted["rf"].transform(X_test).head()

,Tenure,WarehouseToHome,HourSpendOnApp,NumberOfDeviceRegistered,NumberOfAddress,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,CashbackAmount__missing,Complain,PreferredLoginDevice_Computer,PreferredLoginDevice_Mobile Phone,PreferredPaymentMode_Cash on Delivery,PreferredPaymentMode_Credit Card,PreferredPaymentMode_Debit Card,PreferredPaymentMode_E wallet,PreferredPaymentMode_UPI,PreferedOrderCat_Fashion,PreferedOrderCat_Grocery,PreferedOrderCat_Laptop & Accessory,PreferedOrderCat_Missing,PreferedOrderCat_Mobile Phone,PreferedOrderCat_Others,MaritalStatus_Divorced,MaritalStatus_Married,MaritalStatus_Single,SatisfactionScore
4006,17.0,9.0,4.0,4.0,3.0,21.0,6.0,8.0,8.0,181.75,0.0,0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1969,26.0,13.0,3.0,4.0,8.0,18.0,0.0,1.0,3.0,209.34,0.0,1,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4235,5.0,15.0,4.0,4.0,3.0,24.0,2.0,2.0,9.0,169.24,0.0,0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
105,25.0,8.0,2.0,4.0,8.0,15.0,0.0,2.0,2.0,132.12,0.0,0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
2295,4.0,8.0,2.0,3.0,2.0,19.0,8.0,9.0,6.0,198.52,0.0,1,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0


### Аудиторський слід адаптивного імпутера

Частка пропусків — та, яку імпутер **побачив на train**. Задіяно гілки політики:
числова ≤10% → медіана; числова 10–40% → медіана + індикатор (`CashbackAmount`); категоріальна ≤10% → мода; категоріальна >10% → `Missing` (`PreferedOrderCat`).
Імпутація однакова для обох родин — осі «частка пропусків» і «родина моделі» незалежні.


In [14]:
audit = builders["lr"].audit_trail(fitted["lr"])
audit

,feature,group,missing_share,rule_max_share,strategy,fill_value,add_indicator
0,Tenure,numeric_continuous,0.0473,0.1,median,9.0,False
1,WarehouseToHome,numeric_continuous,0.0457,0.1,median,14.0,False
2,HourSpendOnApp,numeric_continuous,0.0437,0.1,median,3.0,False
3,NumberOfDeviceRegistered,numeric_continuous,0.0000,0.1,median,4.0,False
4,NumberOfAddress,numeric_continuous,0.0000,0.1,median,3.0,False
5,OrderAmountHikeFromlastYear,numeric_continuous,0.0462,0.1,median,15.0,False
6,CouponUsed,numeric_continuous,0.0462,0.1,median,1.0,False
7,OrderCount,numeric_continuous,0.0471,0.1,median,2.0,False
8,DaySinceLastOrder,numeric_continuous,0.0548,0.1,median,3.0,False
9,CashbackAmount,numeric_continuous,0.2016,0.4,median,163.58,True


In [15]:
assert audit.equals(builders["rf"].audit_trail(fitted["rf"]))   # той самий результат для обох родин
metadata = builders["lr"].metadata_table(fitted["lr"])
(REPORTS / "metadata_table.md").write_text(metadata.to_markdown(index=False), encoding="utf-8")
metadata

,feature,group,missing_share,rule,additive,partitioning
0,Tenure,numeric_continuous,0.0473,"max_share: 0.10, median",log1p → standard_scaler,winsorize
1,WarehouseToHome,numeric_continuous,0.0457,"max_share: 0.10, median",log1p → standard_scaler,winsorize
2,HourSpendOnApp,numeric_continuous,0.0437,"max_share: 0.10, median",log1p → standard_scaler,winsorize
3,NumberOfDeviceRegistered,numeric_continuous,0.0000,"max_share: 0.10, median",log1p → standard_scaler,winsorize
4,NumberOfAddress,numeric_continuous,0.0000,"max_share: 0.10, median",log1p → standard_scaler,winsorize
5,OrderAmountHikeFromlastYear,numeric_continuous,0.0462,"max_share: 0.10, median",log1p → standard_scaler,winsorize
6,CouponUsed,numeric_continuous,0.0462,"max_share: 0.10, median",log1p → standard_scaler,winsorize
7,OrderCount,numeric_continuous,0.0471,"max_share: 0.10, median",log1p → standard_scaler,winsorize
8,DaySinceLastOrder,numeric_continuous,0.0548,"max_share: 0.10, median",log1p → standard_scaler,winsorize
9,CashbackAmount,numeric_continuous,0.2016,"max_share: 0.40, median + indicator",log1p → standard_scaler,winsorize


## 5. `transform` не переобчислює частку пропусків

Додаємо ще 5% пропусків у `Tenure` на тестовій підвибірці (частка зростає приблизно вдвічі). Імпутер має:
не змінити свій стан, заповнити пропуски **медіаною train** і не додати нових колонок.


In [16]:
imp = fitted["lr"].named_transformers_["numeric_continuous"].named_steps["imputer"]
state_before = dict(imp.audit_["Tenure"])

X_shift = X_test.copy()
rng = np.random.default_rng(0)
extra = rng.choice(np.flatnonzero(X_shift["Tenure"].notna()), size=int(0.05 * len(X_shift)), replace=False)
X_shift.iloc[extra, X_shift.columns.get_loc("Tenure")] = np.nan
print(f"Частка пропусків Tenure: train={state_before['missing_share']:.3f}, test зі зсувом={X_shift['Tenure'].isna().mean():.3f}")

out = imp.transform(X_shift[list(imp.feature_names_in_)])
print("Стан імпутера не змінився:", imp.audit_["Tenure"] == state_before)
print("Розмірність виходу та сама:", out.shape[1] == len(imp.get_feature_names_out()))
print("Заповнено медіаною train:", set(out.loc[X_shift["Tenure"].isna(), "Tenure"]) == {state_before["fill_value"]})

Частка пропусків Tenure: train=0.047, test зі зсувом=0.095
Стан імпутера не змінився: True
Розмірність виходу та сама: True
Заповнено медіаною train: True


Це не дефект, а контракт: у продакшені розподіл майбутніх даних невідомий, тож правило фіксується при навчанні.
Різке зростання частки пропусків — сигнал для **моніторингу** (дрейф даних), а не привід мовчки змінити поведінку препроцесора.


## 6. Протокол верифікації

In [17]:
import os
os.chdir(ROOT)
import run_verification as rv
for path in ("config/config_lr.yaml", "config/config_rf.yaml"):
    rv.test_idempotent(path)
    rv.test_protected_excluded(path)
    print(f"{path}: перевірки пройдено")

config/config_lr.yaml: перевірки пройдено
config/config_rf.yaml: перевірки пройдено
